# BCI Competition IV – Dataset 2b: Exploración y verificación

**Trabajo de Grado — Maestría en Ingeniería, IUE 2026**  
Autor: Juan Carlos Guerrero Sierra

---

Este notebook verifica que los archivos GDF están correctos, visualiza las señales crudas y filtradas, y genera los espectrogramas ERSP de ejemplo para un sujeto.

**Antes de empezar:**
1. Asegúrate de que los 18 archivos GDF están en `data/raw/`
2. Ejecuta `pip install -r requirements.txt` si no lo has hecho

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import mne
mne.set_log_level('WARNING')

from config import *
from preprocessing import verify_dataset, load_raw, apply_filter, extract_epochs, plot_subject_overview, plot_all_subjects_summary
from ersp import compute_ersp_image, generate_ersp_for_subject, plot_ersp_examples, plot_ersp_average

print('Módulos cargados correctamente.')

## 1. Verificación del dataset

Comprobamos que los 18 archivos GDF están presentes.

In [ ]:
ok = verify_dataset()
if ok:
    print('\n✓ Dataset completo. Listo para procesar.')
else:
    print('\n✗ Faltan archivos. Descarga el dataset desde:')
    print('  https://www.bbci.de/competition/iv/download/')

## 2. Inspección de un archivo GDF

Cargamos el sujeto 1 (entrenamiento) y revisamos su contenido.

In [ ]:
SUBJECT = 1   # cambia este número para explorar otros sujetos
SUFFIX  = 'T' # 'T' = entrenamiento (sesiones 1-3)

raw = load_raw(SUBJECT, SUFFIX)

print(f'Sujeto S{SUBJECT:02d}{SUFFIX}')
print(f'  Canales:            {raw.ch_names}')
print(f'  Frecuencia muestreo: {raw.info["sfreq"]} Hz')
print(f'  Duración total:      {raw.n_times / raw.info["sfreq"]:.1f} s')
print(f'  Número de muestras:  {raw.n_times}')

In [ ]:
# Ver eventos disponibles en el archivo
events, event_id = mne.events_from_annotations(raw, verbose=False)
print(f'Eventos encontrados: {event_id}')
print(f'Total de eventos: {len(events)}')
print(f'\nPrimeros 10 eventos:')
for e in events[:10]:
    print(f'  muestra={e[0]}, t={e[0]/raw.info["sfreq"]:.2f}s, tipo={e[2]}')

## 3. Visualización de la señal cruda vs. filtrada

In [ ]:
# Señal cruda
data_raw = raw.get_data() * 1e6  # en µV
t = raw.times

# Señal filtrada (8-30 Hz)
raw_filt = raw.copy()
apply_filter(raw_filt)
data_filt = raw_filt.get_data() * 1e6

fig, axes = plt.subplots(len(raw.ch_names), 2, figsize=(16, 4*len(raw.ch_names)))
if len(raw.ch_names) == 1:
    axes = axes.reshape(1, -1)

# Mostrar solo los primeros 15 segundos
mask = t < 15

for i, ch_name in enumerate(raw.ch_names):
    axes[i,0].plot(t[mask], data_raw[i, mask], color='#2C7BB6', lw=0.6)
    axes[i,0].set_title(f'{ch_name} — Señal cruda')
    axes[i,0].set_ylabel('Amplitud (µV)')
    axes[i,0].spines['top'].set_visible(False)
    axes[i,0].spines['right'].set_visible(False)
    
    axes[i,1].plot(t[mask], data_filt[i, mask], color='#D7191C', lw=0.6)
    axes[i,1].set_title(f'{ch_name} — Filtrada (8–30 Hz)')
    axes[i,1].spines['top'].set_visible(False)
    axes[i,1].spines['right'].set_visible(False)

axes[-1,0].set_xlabel('Tiempo (s)')
axes[-1,1].set_xlabel('Tiempo (s)')
plt.suptitle(f'S{SUBJECT:02d} — Comparación señal cruda vs. filtrada (primeros 15 s)', 
             fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Extracción de épocas y estadísticas

In [ ]:
epochs = extract_epochs(raw_filt.copy())

print(f'Épocas extraídas:')
print(f'  Total válidas:  {len(epochs)}')
print(f'  Izquierda (0):  {len(epochs["left"]) if "left" in epochs.event_id else 0}')
print(f'  Derecha  (1):   {len(epochs["right"]) if "right" in epochs.event_id else 0}')
print(f'  Ventana:        {EPOCH_TMIN} a {EPOCH_TMAX} s')
print(f'  Forma por época: {epochs.get_data().shape}  (épocas × canales × muestras)')

In [ ]:
# Potencial evocado por clase (ERP)
fig, axes = plt.subplots(1, len(epochs.ch_names), figsize=(5*len(epochs.ch_names), 4))
if len(epochs.ch_names) == 1:
    axes = [axes]

for i, ch in enumerate(epochs.ch_names):
    ax = axes[i]
    for cls, color, label in [('left','#2C7BB6','Izquierda'), ('right','#D7191C','Derecha')]:
        if cls in epochs.event_id:
            ep = epochs[cls].get_data(picks=[i])[:,0,:] * 1e6
            ax.plot(epochs.times, ep.mean(0), color=color, label=label, lw=1.5)
            ax.fill_between(epochs.times, 
                            ep.mean(0)-ep.std(0), 
                            ep.mean(0)+ep.std(0), 
                            color=color, alpha=0.15)
    ax.axvline(0, color='k', lw=0.8, ls='--', label='Onset')
    ax.axhline(0, color='gray', lw=0.4, ls=':')
    ax.set_title(f'Canal {ch}')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('µV' if i==0 else '')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(f'S{SUBJECT:02d} — ERP promedio por clase (media ± DE)', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Espectrograma ERSP de ejemplo

Calculamos el espectrograma ERSP de un ensayo individual y lo visualizamos.

In [ ]:
from scipy.signal import stft as scipy_stft

# Tomar el primer ensayo de cada clase, canal C3
ch_idx = 0  # C3
bl_end = int(abs(EPOCH_TMIN) * SFREQ)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cls_list = [('left', 0, 'Izquierda', '#2C7BB6'), ('right', 1, 'Derecha', '#D7191C')]

for row, (cls_name, cls_label, cls_display, color) in enumerate(cls_list):
    if cls_name not in epochs.event_id:
        continue
    ep_data = epochs[cls_name].get_data()  # (N, C, T)
    signal   = ep_data[0, ch_idx, :]       # primer ensayo, canal C3
    baseline = signal[:bl_end]
    
    ersp_img = compute_ersp_image(signal, baseline, sfreq=SFREQ)
    
    # Señal temporal
    ax = axes[row, 0]
    ax.plot(epochs.times, signal*1e6, color=color, lw=0.8)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.axvspan(EPOCH_TMIN, 0, alpha=0.1, color='gray', label='Línea base')
    ax.set_title(f'Señal EEG — {cls_display} (C3, ensayo 1)')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Amplitud (µV)')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Espectrograma ERSP
    ax = axes[row, 1]
    im = ax.imshow(
        ersp_img, aspect='auto', origin='lower', cmap='RdYlBu_r',
        extent=[EPOCH_TMIN, EPOCH_TMAX, ERSP_FMIN, ERSP_FMAX],
        vmin=0, vmax=1
    )
    ax.axvline(0, color='white', lw=1.0, ls='--')
    ax.axhspan(8, 13, alpha=0.15, color='cyan')    # banda mu
    ax.axhspan(14, 30, alpha=0.10, color='yellow') # banda beta
    ax.set_title(f'Espectrograma ERSP — {cls_display} (C3, ensayo 1)\n'
                 f'(cian=mu, amarillo=beta)')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Frecuencia (Hz)')
    plt.colorbar(im, ax=ax, label='ERSP normalizado [0,1]')

plt.suptitle(f'S{SUBJECT:02d} — Señal EEG y espectrograma ERSP por clase', 
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Resumen de épocas para todos los sujetos

Verificamos cuántos ensayos válidos hay por sujeto (útil para detectar sujetos problemáticos).

In [ ]:
# Este proceso puede tardar varios minutos (carga y procesa los 9 sujetos)
# Cambia suffix a 'E' para ver las sesiones de evaluación
plot_all_subjects_summary(suffix='T')

## 7. Preprocesamiento completo de un sujeto

Si todo se ve bien, ejecuta el preprocesamiento completo desde terminal:

```bash
# Un sujeto con visualización
python src/preprocessing.py --subject 1 --plot

# Todos los sujetos
python src/preprocessing.py --subject all --suffix both
```

Luego genera los espectrogramas:

```bash
python src/ersp.py --subject all --plot
```

Y entrena los modelos:

```bash
python src/train.py --model spectnet --all_subjects
python src/train.py --model eegnet --all_subjects
python src/train.py --model shallowconvnet --all_subjects
```

In [ ]:
print('Exploración completada.')
print(f'Figuras guardadas en: {FIGURES_DIR}')